In [99]:
from pathlib import Path
import duckdb
import pandas as pd

class DataHubReader:
    def __init__(self, db_name="bronze.db"):
        """
        Stellt Verbindung zur DuckDB her.
        Erwartet dieselbe Ordnerstruktur wie DataHub.
        """
        base_path = Path.cwd()
        db_path = base_path.parent.parent / "data" /  db_name

        if not db_path.exists():
            raise FileNotFoundError(f"Datenbank nicht gefunden: {db_path}")

        self.con = duckdb.connect(str(db_path))

    def _fetch_table(self, table_name: str) -> pd.DataFrame:
        """Generische Methode zum Laden einer Tabelle."""
        return self.con.execute(f"SELECT * FROM {table_name}").df()

    def get_financials(self) -> pd.DataFrame:
        """Yahoo Finance Financials"""
        return self._fetch_table("bronze_financials")

    def get_wikidata(self) -> pd.DataFrame:
        """Wikidata Unternehmensdaten"""
        return self._fetch_table("bronze_wikidata")

    def get_GMD(self) -> pd.DataFrame:
        """Global Macro Database"""
        return self._fetch_table("bronze_gmd")

    def get_interest(self) -> pd.DataFrame:
        """IMF Zinsdaten"""
        return self._fetch_table("bronze_interest")

    def get_commodity(self) -> pd.DataFrame:
        """IMF Commodity Daten"""
        return self._fetch_table("bronze_commodities")

    def close(self):
        """Schließt die Datenbankverbindung."""
        self.con.close()

In [100]:
df = DataHubReader().get_wikidata()
df

,ticker,company_qid,item_description,value,label,ingested_at
0,ALATA.PA,Q753684,isin,FR0010478248,nan,2026-04-10 08:14:08.375809
1,ALATA.PA,Q753684,isin,US4659401040,nan,2026-04-10 08:14:08.375809
2,ALCOX.PA,Q906797,isin,FR0013018124,nan,2026-04-10 08:14:08.375809
3,ALPUL.PA,Q921944,isin,FR0012419307,nan,2026-04-10 08:14:08.375809
4,ALCGM.PA,Q1052675,isin,FR0000053506,nan,2026-04-10 08:14:08.375809
...,...,...,...,...,...,...
36593,ZEST.MI,Q93097534,owned_by,nan,nan,2026-04-10 08:14:08.375809
36594,TPRO.MI,Q111142424,owned_by,Q112479322,Giuseppe Crippa,2026-04-10 08:14:08.375809
36595,ZUC.MI,Q115167588,owned_by,nan,nan,2026-04-10 08:14:08.375809
36596,TISG.MI,Q129823767,owned_by,nan,nan,2026-04-10 08:14:08.375809


In [101]:
from datetime import datetime
import numpy as np
class WikidatatoSilver():
    def __init__(self):
        pass
    def correct_nan(self,df):
        df = df.copy()
        df.loc[df['value'] == 'nan', 'value'] = pd.NA
        return df
    def get_age(self, df):
        fj = df[df['item_description'] == 'founding_year'].drop_duplicates(subset=["company_qid"]).copy()
        current_year = datetime.now().year
        fj["value"] = current_year - pd.to_datetime(fj["value"], errors="coerce").dt.year 
        fj['item_description'] = 'company_age'
        return fj
    def replace_qid_labels_with_nan(self, df):
        df = df.copy()
        mask = df["label"].astype(str).str.match(r"^Q\d+$", na=False)
        df.loc[mask, "label"] = pd.NA
        return df
    def replace_qids_with_ticker(self, df, only_replaced: bool = False):
        df = df.copy()
        mapping = (
            df[["company_qid", "ticker"]]
            .dropna(subset=["company_qid", "ticker"])
            .drop_duplicates(subset=["company_qid", "ticker"])
            .drop_duplicates(subset=["company_qid"], keep="first")
            .set_index("company_qid")["ticker"]
        )
        # Identifiziere QIDs
        mask = df["value"].astype(str).str.match(r"^Q\d+$", na=False)
        # Originalwerte speichern für Vergleich
        original_values = df.loc[mask, "value"].copy()
        # Mapping durchführen
        replaced_values = (original_values.astype(str).str.strip().map(mapping))
        # Ersetze nur dort, wo Mapping existiert
        df.loc[mask, "value"] = replaced_values.fillna(original_values)
        if only_replaced:
            # Nur Zeilen behalten, wo tatsächlich eine Ersetzung passiert ist
            replaced_mask = mask & replaced_values.notna()
            return df.loc[replaced_mask].copy()

        return df
    def drop_qid_relations(self, df):
        df = df.copy()

        # Relevante Item_Descriptions
        target_items = ["subsidiaries", "investments", "owned_by"]

        # Bedingung: richtige Kategorie + Value ist QID
        mask = (
            df["item_description"].isin(target_items) &
            df["value"].astype(str).str.startswith("Q", na=False)
        )

        # Entfernen der Zeilen
        df = df.loc[~mask].copy()

        return df
    def filter_isin_founding(self, df):
        df = df[df['item_description'] != 'isin']
        df = df[df['item_description'] != 'founding_year']
        df = df[df['item_description'] != 'industries']
        df = df[df['item_description'] != 'operating_area']
        df = df[df['item_description'] != 'location']

        return df

    def run(self, df):
        nan = self.correct_nan(df)
        nan = nan.dropna()
        nan = self.filter_isin_founding(nan)
        with_tickers = self.replace_qids_with_ticker(nan)
        label_cleaning = self.replace_qid_labels_with_nan(with_tickers)
        value_cleaning = self.drop_qid_relations(label_cleaning)
        return value_cleaning
    

    

In [102]:
pd.set_option("display.max_rows", 1000)
erg = WikidatatoSilver().run(df)
erg.to_csv("wikidata.csv", encoding="utf-8")
#erg['value'].value_counts()
erg[erg['item_description'] == "operating_area" ]["label"].value_counts()

Series([], Name: count, dtype: int64)

In [96]:
erg['item_description'].unique()

array(['products', 'instance_of', 'owned_by', 'part_of', 'investments',
       'subsidiaries'], dtype=object)

In [48]:
pd.set_option("display.max_rows", None)
def classify_value_type(df):
    df = df.copy()
    s = df["value"].astype("string").str.strip()

    is_qid = s.str.fullmatch(r"Q\d+", na=False)
    is_isin = s.str.fullmatch(r"[A-Z]{2}[A-Z0-9]{9}\d", na=False)
    is_date = pd.to_datetime(s, errors="coerce", utc=True).notna()

    df["value_type"] = "ticker"

    df.loc[s.isna() | (s == ""), "value_type"] = "missing"
    df.loc[is_qid, "value_type"] = "qid"
    df.loc[is_isin, "value_type"] = "isin"
    df.loc[is_date, "value_type"] = "date"

    return df
erg = WikidatatoSilver().run(df)
s = erg["value"].astype("string").str.strip()

is_qid = s.str.fullmatch(r"Q\d+", na=False)
is_isin = s.str.fullmatch(r"[A-Z]{2}[A-Z0-9]{9}\d", na=False)
is_date = pd.to_datetime(s, errors="coerce").notna()

is_ticker = ~(is_qid | is_isin | is_date) & s.notna() & (s != "")

print("Ticker-Kandidaten:", is_ticker.sum())


C:\Users\Konra\AppData\Local\Temp\ipykernel_8932\1446774560.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  is_date = pd.to_datetime(s, errors="coerce").notna()


Ticker-Kandidaten: 342
